# Customer Personality Analysis & Segmentation
### Unsupervised Machine Learning Pipeline using K-Means and PCA

Customer Personality Analysis provides detailed analysis of a company's ideal customers. It helps a business to better understand its customers and makes it easier for them to modify products according to the specific needs, behaviors, and concerns of different types of customers.

## 1. Import Libraries and Load Dataset

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Set styling
plt.style.use('seaborn-v0_8-whitegrid')
df = pd.read_csv('customer_segmentation_data.csv')
df.head()

## 2. Exploratory Data Analysis & Feature Engineering
- Calculate `Age = 2026 - Year_Birth`
- Calculate `Total_Spent` across Wines, Fruits, Meat, Fish, Sweets, and Gold
- Compute `Children = Kidhome + Teenhome`
- Compute `Total_Purchases = Web + Catalog + Store`

In [2]:
spend_cols = ['MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds']
df['Total_Spent'] = df[spend_cols].sum(axis=1)
df['Children'] = df['Kidhome'] + df['Teenhome']
df['Total_Purchases'] = df['NumDealsPurchases'] + df['NumWebPurchases'] + df['NumCatalogPurchases'] + df['NumStorePurchases']

print("Dataset Summary:")
print(df[['Income', 'Age', 'Total_Spent', 'Children', 'Total_Purchases']].describe().round(2))

## 3. Data Scaling and Dimensionality Reduction (PCA)
Standardizing features to zero mean and unit variance before running clustering.

In [3]:
features = [
    'Income', 'Age', 'Total_Spent', 'Children',
    'MntWines', 'MntMeatProducts', 'MntGoldProds',
    'NumDealsPurchases', 'NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases',
    'NumWebVisitsMonth'
]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[features])

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
print(f"Explained Variance: {pca.explained_variance_ratio_}")

## 4. Elbow Method and Silhouette Analysis for Optimal K

In [4]:
inertias = []
sil_scores = []
K_range = range(2, 8)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

fig, ax1 = plt.subplots(1, 2, figsize=(14, 5))
ax1[0].plot(K_range, inertias, 'bo-')
ax1[0].set_title('Elbow Method (Inertia vs. K)')
ax1[0].set_xlabel('Clusters (k)')
ax1[0].set_ylabel('Inertia')

ax1[1].plot(K_range, sil_scores, 'rs-')
ax1[1].set_title('Silhouette Score vs. K')
ax1[1].set_xlabel('Clusters (k)')
ax1[1].set_ylabel('Silhouette Score')
plt.show()

## 5. Fit K-Means (k=4) and Visualize Clusters in Latent PCA Space

In [5]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)
df['PCA1'] = X_pca[:, 0]
df['PCA2'] = X_pca[:, 1]

plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='PCA1', y='PCA2', hue='Cluster', palette='tab10', alpha=0.8)
plt.title('Customer Segments in 2D PCA Space')
plt.show()

## 6. Strategic Persona Profiling & Business Recommendations

In [6]:
cluster_names = {
    0: 'Stars / Luxury Connoisseurs',
    1: 'Frugal / Budget Seekers',
    2: 'Prudent Middle-Class',
    3: 'Emerging Tech Professionals'
}
df['Persona'] = df['Cluster'].map(cluster_names)
profile = df.groupby('Persona')[['Income', 'Total_Spent', 'Children', 'NumDealsPurchases', 'NumWebPurchases', 'NumStorePurchases', 'Response']].mean().round(2)
print(profile)